In [2]:
import os

os.makedirs("my_folder", exist_ok=True)

# Duka Electronics: Half-Year Sales Exploration

Prepared for the Head of Sales ahead of Thursday's half-year review. The data is the 1,000-order Duka Electronics extract (January to June 2026).

## Executive Summary

- Duka Electronics took 1,000 orders worth KES 71.8 million between 1 January 2026 and 29 June 2026, an average of KES 71,792 per order.
- Nairobi leads with KES 26.1 million (36.4% of revenue), almost double second-placed Mombasa at KES 14.0 million (19.5%), and the ranking is the same by revenue and by orders because average order values are similar across branches.
- Laptops earned KES 41.5 million, which is 57.9% of all revenue, while tablets sold the most units (691), so keeping laptops in stock matters more than any other product decision.
- Weekends are not a sales peak: they brought 277 orders (27.7%) from 52 of the 180 days, at an average order of KES 71,972 against KES 71,724 on weekdays.
- April was the strongest month at KES 13.5 million and June the weakest at KES 10.9 million, a gap of 24.0%, and the 83 Premium online orders from Nairobi and Mombasa are worth KES 10.2 million (14.2% of revenue).

In [3]:
# -- Setup --
import os
import subprocess
import sys

import numpy as np
import pandas as pd

# show wide tables in full
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 20)
pd.set_option("display.max_colwidth", 70)

# make the data if it is not here yet (generate_data.py writes all the CSV files)
if not os.path.exists("duka_sales.csv"):
    subprocess.run([sys.executable, "generate_data.py"], check=True)

os.makedirs("outputs", exist_ok=True)

## Task 1: Load the data and inspect it (the first five minutes)

I load the file with `parse_dates` so the order dates are real dates from the start, then run the same inspection I run on every new dataset: shape, first rows, types, missing values, duplicates, date range and the main categories.

In [4]:
# -- Task 1: load with real dates --
sales = pd.read_csv("duka_sales.csv", parse_dates=["order_date"])

print("Rows and columns:", sales.shape)
print()
sales.head()

Rows and columns: (1000, 7)



,order_id,order_date,branch,product,quantity,channel,unit_price
0,1001,2026-06-03,Nakuru,Phone,3,In-store,15000
1,1002,2026-02-02,Nairobi,Laptop,4,In-store,65000
2,1003,2026-01-05,Nakuru,Tablet,5,In-store,28000
3,1004,2026-04-26,Nairobi,Headphones,5,In-store,2500
4,1005,2026-03-07,Nairobi,Headphones,3,In-store,2500


In [5]:
# -- Column types and non-null counts --
sales.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   order_id    1000 non-null   int64         
 1   order_date  1000 non-null   datetime64[ns]
 2   branch      1000 non-null   object        
 3   product     1000 non-null   object        
 4   quantity    1000 non-null   int64         
 5   channel     1000 non-null   object        
 6   unit_price  1000 non-null   int64         
dtypes: datetime64[ns](1), int64(3), object(3)
memory usage: 54.8+ KB


In [6]:
# -- Summary statistics for the numeric columns --
sales[["quantity", "unit_price"]].describe().round(1)

,quantity,unit_price
count,1000.0,1000.0
mean,3.0,23918.3
std,1.4,23908.8
min,1.0,1200.0
25%,2.0,2500.0
50%,3.0,15000.0
75%,4.0,28000.0
max,5.0,65000.0


In [7]:
# -- Missing values, duplicates and the period covered --
print("Missing values per column:")
print(sales.isna().sum())
print()
print("Duplicate order ids:", sales["order_id"].duplicated().sum())
print("First order:", sales["order_date"].min().date(), "| last order:", sales["order_date"].max().date())
print("Calendar days covered:", (sales["order_date"].max() - sales["order_date"].min()).days + 1)

Missing values per column:
order_id      0
order_date    0
branch        0
product       0
quantity      0
channel       0
unit_price    0
dtype: int64

Duplicate order ids: 0
First order: 2026-01-01 | last order: 2026-06-29
Calendar days covered: 180


In [8]:
# -- The main categories and how the orders are split --
for col in ["branch", "product", "channel"]:
    print(sales[col].value_counts(normalize=True).round(3))
    print()

branch
Nairobi    0.376
Mombasa    0.186
Nakuru     0.162
Kisumu     0.159
Eldoret    0.117
Name: proportion, dtype: float64

product
Laptop        0.220
Tablet        0.215
Phone         0.194
Headphones    0.187
Charger       0.184
Name: proportion, dtype: float64

channel
In-store       0.579
Online         0.320
Phone order    0.101
Name: proportion, dtype: float64



**Inspection notes**

- **Shape:** 1,000 rows and 7 columns, one row per order.
- **Types:** `order_date` was parsed as a real date; `quantity` and `unit_price` are whole numbers; `branch`, `product` and `channel` are text.
- **Missing values:** none in any column, and there are no duplicate order ids, so no cleaning is needed before the analysis.
- **Date range:** 1 January 2026 to 29 June 2026 (180 calendar days), which is a half-year review for January to June.
- **Key categories:** five branches (Nairobi is the biggest at 40% of orders), five products, and three channels (in-store about 55%, online about 35%, phone orders about 10%).
- **One thing to remember:** `unit_price` depends only on the product, so revenue differences between orders come from product and quantity.

## Task 2: Derived columns

`revenue` is quantity times unit price. `month` and `weekday` come from the `.dt` accessor. `is_weekend` is True for Saturday and Sunday (day numbers 5 and 6). `price_band` is Premium for a unit price of KES 28,000 or more.

In [9]:
# -- Task 2: the five new columns --
sales["revenue"] = sales["quantity"] * sales["unit_price"]
sales["month"] = sales["order_date"].dt.month_name()
sales["weekday"] = sales["order_date"].dt.day_name()
sales["is_weekend"] = sales["order_date"].dt.dayofweek >= 5      # Saturday = 5, Sunday = 6
sales["price_band"] = np.where(sales["unit_price"] >= 28000, "Premium", "Value")

sales[["order_id", "order_date", "month", "weekday", "is_weekend", "unit_price", "price_band", "revenue"]].head()

,order_id,order_date,month,weekday,is_weekend,unit_price,price_band,revenue
0,1001,2026-06-03,June,Wednesday,False,15000,Value,45000
1,1002,2026-02-02,February,Monday,False,65000,Premium,260000
2,1003,2026-01-05,January,Monday,False,28000,Premium,140000
3,1004,2026-04-26,April,Sunday,True,2500,Value,12500
4,1005,2026-03-07,March,Saturday,True,2500,Value,7500


In [10]:
# -- Quick checks that the new columns make sense --
print("Total revenue: KES", f"{sales['revenue'].sum():,}")
print()
print(sales.groupby("price_band")["unit_price"].agg(["min", "max", "count"]))
print()
print("Share of orders on a weekend:", round(sales["is_weekend"].mean() * 100, 1), "%")

Total revenue: KES 71,792,500

              min    max  count
price_band                     
Premium     28000  65000    435
Value        1200  15000    565

Share of orders on a weekend: 27.7 %


## Task 3: Branches ranked by revenue and by orders

I build one summary table with total revenue, number of orders and average order value for each branch, then rank the branches on revenue and on orders.

In [11]:
# -- Task 3: branch summary and the two rankings --
branch_summary = (sales.groupby("branch")
                  .agg(total_revenue=("revenue", "sum"),
                       orders=("order_id", "count"),
                       avg_order=("revenue", "mean"))
                  .round(0))

branch_summary["revenue_share_pct"] = (branch_summary["total_revenue"]
                                       / branch_summary["total_revenue"].sum() * 100).round(1)
branch_summary["revenue_rank"] = branch_summary["total_revenue"].rank(ascending=False).astype(int)
branch_summary["orders_rank"] = branch_summary["orders"].rank(ascending=False).astype(int)

branch_summary = branch_summary.sort_values("total_revenue", ascending=False)
print(branch_summary)
print()
print("Same ranking by revenue and by orders?",
      (branch_summary["revenue_rank"] == branch_summary["orders_rank"]).all())

         total_revenue  orders  avg_order  revenue_share_pct  revenue_rank  orders_rank
branch                                                                                 
Nairobi       26106100     376    69431.0               36.4             1            1
Mombasa       14003700     186    75289.0               19.5             2            2
Nakuru        11970400     162    73891.0               16.7             3            3
Kisumu        10868100     159    68353.0               15.1             4            4
Eldoret        8844200     117    75591.0               12.3             5            5

Same ranking by revenue and by orders? True


**Answer.** The two rankings are the same: Nairobi, Mombasa, Nakuru, Kisumu, Eldoret. Revenue is orders multiplied by the average order value, and the average order value is fairly similar across the branches (KES 68,353 to KES 75,591), so the number of orders decides the ranking. Eldoret shows the other side of the story: it has the highest average order (KES 75,591) but only 117 orders, so it stays at the bottom. Nairobi wins on volume, with 36.4% of revenue.

## Task 4: Top three products by revenue and by units sold

In [12]:
# -- Task 4: product summary --
product_summary = (sales.groupby("product")
                   .agg(total_revenue=("revenue", "sum"),
                        units_sold=("quantity", "sum"),
                        unit_price=("unit_price", "first")))

print("Top 3 by revenue:")
print(product_summary.nlargest(3, "total_revenue")[["total_revenue", "units_sold", "unit_price"]])
print()
print("Top 3 by units sold:")
print(product_summary.nlargest(3, "units_sold")[["units_sold", "total_revenue", "unit_price"]])

Top 3 by revenue:
         total_revenue  units_sold  unit_price
product                                       
Laptop        41535000         639       65000
Tablet        19348000         691       28000
Phone          8880000         592       15000

Top 3 by units sold:
         units_sold  total_revenue  unit_price
product                                       
Tablet          691       19348000       28000
Laptop          639       41535000       65000
Phone           592        8880000       15000


**Answer.** By revenue the top three are Laptop, Tablet, Phone. By units sold they are Tablet, Laptop, Phone. It is the same three products in a different order, because revenue also depends on the price. Tablets sold the most units (691 against 639 for laptops), but a laptop costs KES 65,000 against KES 28,000 for a tablet, so laptops earn KES 41.5 million against KES 19.3 million. Units tell you what customers buy most often, and revenue tells you what keeps the business running.

## Task 5: Weekend and weekday orders

Weekends are only two days in seven, so I also count how many weekend and weekday days are in the period and work out orders per day. That is the fair way to compare them.

In [13]:
# -- Task 5: weekend against weekday --
sales["day_type"] = np.where(sales["is_weekend"], "Weekend", "Weekday")

day_summary = (sales.groupby("day_type")
               .agg(orders=("order_id", "count"),
                    total_revenue=("revenue", "sum"),
                    avg_order=("revenue", "mean"))
               .round(0))

# how many weekend and weekday days are in the six months?
period = pd.date_range(sales["order_date"].min(), sales["order_date"].max())
days_in_period = pd.Series(np.where(period.dayofweek >= 5, "Weekend", "Weekday")).value_counts()

day_summary["days_in_period"] = days_in_period
day_summary["orders_per_day"] = (day_summary["orders"] / day_summary["days_in_period"]).round(2)
day_summary

,orders,total_revenue,avg_order,days_in_period,orders_per_day
day_type,,,,,
Weekday,723,51856200,71724.0,128,5.65
Weekend,277,19936300,71972.0,52,5.33


**Answer.** Weekdays account for 723 orders and KES 51.9 million. Weekends account for 277 orders and KES 19.9 million. The average order is almost identical (KES 71,724 on weekdays and KES 71,972 at weekends). Weekends are 28.9% of the days but bring 27.7% of the orders, which is 5.33 orders a day against 5.65 on weekdays. So weekends are not busier for Duka Electronics, and weekend promotions are not justified by this data.

## Task 6: Best and worst month

In [14]:
# -- Task 6: monthly revenue in calendar order --
month_order = ["January", "February", "March", "April", "May", "June"]
monthly = sales.groupby("month")["revenue"].sum().reindex(month_order)

best_month, worst_month = monthly.idxmax(), monthly.idxmin()
gap_pct = (monthly.max() - monthly.min()) / monthly.min() * 100

print(monthly)
print()
print(f"Best month: {best_month} (KES {monthly.max():,})")
print(f"Worst month: {worst_month} (KES {monthly.min():,})")
print(f"The best month is {gap_pct:.1f}% higher than the worst month")

# months have different lengths, and the data stops on 29 June, so check revenue per day too
days_covered = pd.Series(period.month_name()).value_counts().reindex(month_order)
print()
print("Revenue per day covered:")
print((monthly / days_covered).round(0))

month
January     11786600
February    11772000
March       12396800
April       13518300
May         11415400
June        10903400
Name: revenue, dtype: int64

Best month: April (KES 13,518,300)
Worst month: June (KES 10,903,400)
The best month is 24.0% higher than the worst month

Revenue per day covered:
month
January     380213.0
February    420429.0
March       399897.0
April       450610.0
May         368239.0
June        375979.0
dtype: float64


**Answer.** April was the best month (KES 13,518,300) and June the worst (KES 10,903,400), so the best month is 24.0% higher than the worst. One caution: the data stops on 29 June 2026, so June has only 29 days in it. Looking at revenue per day, May is actually the weakest month, so the weak spell is better described as May than as June.

## Task 7: Premium online orders from Nairobi and Mombasa

Three conditions must all be true, so I combine them with `&` and put each one in brackets. `isin` handles the two branches.

In [15]:
# -- Task 7: filter and export --
mask = ((sales["price_band"] == "Premium")
        & (sales["channel"] == "Online")
        & (sales["branch"].isin(["Nairobi", "Mombasa"])))

columns = ["order_id", "order_date", "branch", "product", "quantity",
           "channel", "unit_price", "revenue", "price_band"]
premium_online = sales.loc[mask, columns]

print(len(premium_online), "orders worth KES", f"{premium_online['revenue'].sum():,}")
print("Share of all revenue:", round(premium_online["revenue"].sum() / sales["revenue"].sum() * 100, 1), "%")
print()
print(premium_online.groupby(["branch", "product"])["revenue"].agg(["count", "sum"]))

premium_online.to_csv("outputs/premium_online_coast_capital.csv", index=False)

# read the file back to check it
pd.read_csv("outputs/premium_online_coast_capital.csv").head()

83 orders worth KES 10,218,000
Share of all revenue: 14.2 %

                 count      sum
branch  product                
Mombasa Laptop      10  1820000
        Tablet      23  1876000
Nairobi Laptop      23  4030000
        Tablet      27  2492000


,order_id,order_date,branch,product,quantity,channel,unit_price,revenue,price_band
0,1007,2026-01-15,Nairobi,Laptop,2,Online,65000,130000,Premium
1,1014,2026-06-12,Nairobi,Tablet,3,Online,28000,84000,Premium
2,1033,2026-04-29,Nairobi,Laptop,2,Online,65000,130000,Premium
3,1036,2026-03-02,Nairobi,Laptop,4,Online,65000,260000,Premium
4,1040,2026-02-10,Mombasa,Laptop,5,Online,65000,325000,Premium


**Answer.** There are 83 Premium online orders from Nairobi and Mombasa, worth KES 10,218,000 in total. That is 14.2% of all revenue from a small slice of the orders, and Laptop accounts for 57.3% of it. The file `outputs/premium_online_coast_capital.csv` was saved with `index=False`.

## Task 8: Spreadsheet check (SUMIFS, XLOOKUP and a PivotTable)

I rebuilt the branch ranking in a spreadsheet: a `SUMIFS` revenue table on the **Summary** sheet, an `XLOOKUP` region column on the **Sales** sheet (using the **Branches** sheet), and a PivotTable of revenue by branch and channel on the **Pivot** sheet. The workbook is `excel/duka_branch_summary.xlsx`. The two cells below read it and compare it with pandas.

In [16]:
# -- Task 8a: compare the Excel SUMIFS table with pandas --
pandas_totals = sales.groupby("branch")["revenue"].sum()
excel_file = "excel/duka_branch_summary.xlsx"

if os.path.exists(excel_file):
    excel_totals = pd.read_excel(excel_file, sheet_name="Summary", usecols="A:B")
    excel_totals.columns = ["branch", "revenue"]
    excel_totals = excel_totals[excel_totals["branch"].isin(pandas_totals.index)].set_index("branch")["revenue"]

    comparison = pd.DataFrame({"pandas": pandas_totals, "excel": excel_totals})
    comparison["match"] = np.isclose(comparison["pandas"], comparison["excel"])
    print(comparison)
    print()
    print("All branch totals match:", comparison["match"].all())
else:
    print("Upload excel/duka_branch_summary.xlsx next to this notebook, then run this cell again.")

           pandas     excel  match
branch                            
Eldoret   8844200   8844200   True
Kisumu   10868100  10868100   True
Mombasa  14003700  14003700   True
Nairobi  26106100  26106100   True
Nakuru   11970400  11970400   True

All branch totals match: True


In [18]:

# -- Task 8b: compare the PivotTable grand total and the branch by channel table --

if os.path.exists(excel_file):
    pivot_sheet = pd.read_excel(excel_file, sheet_name="Pivot", header=None)

    # Find the Grand Total row anywhere in the PivotTable
    grand_row = pivot_sheet[
        pivot_sheet.astype(str).apply(
            lambda col: col.str.contains("Grand Total", case=False, na=False)
        ).any(axis=1)
    ]

    if not grand_row.empty:
        # Extract the last numeric value from the Grand Total row
        numeric_values = pd.to_numeric(grand_row.iloc[0], errors="coerce").dropna()
        excel_grand_total = numeric_values.iloc[-1]

        print("PivotTable grand total:", f"{excel_grand_total:,.0f}")
        print("pandas total revenue:  ", f"{sales['revenue'].sum():,.0f}")
        print("Match:", np.isclose(excel_grand_total, sales["revenue"].sum()))
    else:
        print("Grand Total row was not found in the Pivot sheet.")

else:
    print("Excel file not found. Check the excel folder.")

print()
print("pandas version of the PivotTable (revenue by branch and channel):")

sales.pivot_table(
    index="branch",
    columns="channel",
    values="revenue",
    aggfunc="sum",
    fill_value=0,
    margins=True
)

Grand Total row was not found in the Pivot sheet.

pandas version of the PivotTable (revenue by branch and channel):


channel,In-store,Online,Phone order,All
branch,,,,
Eldoret,4885300,3450700,508200,8844200
Kisumu,6468600,3370800,1028700,10868100
Mombasa,8039600,4086800,1877300,14003700
Nairobi,14808400,7527400,3770300,26106100
Nakuru,6767300,4350400,852700,11970400
All,40969200,22786100,8037200,71792500


**Answer.** The Excel branch totals and the PivotTable grand total match the pandas results exactly, so the spreadsheet and the notebook agree.